# DeepEval Hallucination Evaluation

The purpose of this notebook is to run test cases of hallucination evaluations on datasets.

## Setup

In [17]:
import os
import json
import pandas as pd
import numpy as np
import random
import sys
import io
import copy
import anthropic
from tqdm import tqdm
from dotenv import load_dotenv
import concurrent.futures
import time
import traceback
from deepeval import evaluate
from deepeval.metrics import HallucinationMetric
from deepeval.test_case import LLMTestCase
from deepeval.models.base_model import DeepEvalBaseLLM
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

## LLM Link

In [18]:
load_dotenv()
anthropic_key = os.getenv("ANTHROPIC_KEY")

In [19]:
# define a custom claude model for deepeval
class Claude(DeepEvalBaseLLM):
    """Class to implement Claude model for DeepEval"""
    def __init__(self, model, api_key):
        self.model = model
        self.api_key = api_key
        self.client = anthropic.Anthropic(api_key=self.api_key)

    def load_model(self):
        return self.model

    def generate(self, prompt, max_tokens=4096):
        response = self.client.messages.create(
            model=self.model,
            max_tokens=max_tokens,
            messages=[{"role": "user", "content": prompt}]
        )
        return response.content[0].text

    async def a_generate(self, prompt, max_tokens=4096):
        response = self.client.messages.create(
            model=self.model,
            max_tokens=max_tokens,
            messages=[{"role": "user", "content": prompt}]
        )
        return response.content[0].text

    def get_model_name(self):
        return "Claude Model"

In [20]:
# set up the claude model
claude_model = "claude-3-5-haiku-20241022"
claude_instance = Claude(model=claude_model, api_key=anthropic_key)

## Data

In [21]:
# load json dataset
file_path = "../../data/Hallucination/qa_data.json"
with open(file_path, "r", encoding="utf-8") as file:
    data = [json.loads(line) for line in file]

In [41]:
random.seed(42)

records = []
hallucination_flags = []
answers = []

# Sample 1000 random entries from data
sampled_entries = random.sample(data, 1000)

for entry in sampled_entries:
    is_hallucinated = random.choice([True, False])
    actual_output = entry["hallucinated_answer"] if is_hallucinated else entry["right_answer"]    
    records.append({
        "input": entry["question"],
        "actual_output": actual_output,
        "context": entry["knowledge"]
    })
    hallucination_flags.append(1 if is_hallucinated else 0)
    answers.append(actual_output)

# Convert to DataFrame
df = pd.DataFrame(records).reset_index()

hallucination_df = pd.DataFrame({
    "index": df["index"],
    "hallucinated": hallucination_flags,
    "actual_output": answers
})

In [42]:
hallucination_df.hallucinated.value_counts()

hallucinated
0    534
1    466
Name: count, dtype: int64

## DeepEval Evaluation

In [12]:
# creating test cases for evaluation
test_cases = [
    (row["index"], LLMTestCase(
        input=row["input"],
        actual_output=row["actual_output"],
        context=[row["context"]]
    ))
    for _, row in df.iterrows()
]

In [ ]:
def process_test_case(index, test_case, hallucination_metric):
    try:
        result = evaluate(
            test_cases=[test_case],
            metrics=[hallucination_metric],
            write_cache=False,
            print_results=False,
            display=None,
            show_indicator=False
        )
        return (index, result)
    except Exception as e:
        return (index, f"Error in test case {index}: {str(e)}")

start_time = time.time()

try:
    hallucination_metric = HallucinationMetric(model=claude_instance, threshold=0.5, async_mode=True)

    with concurrent.futures.ThreadPoolExecutor(max_workers=min(5, len(test_cases))) as executor:
        futures = [
            executor.submit(process_test_case, idx, case, hallucination_metric)
            for idx, case in test_cases
        ]
        results = [f.result() for f in concurrent.futures.as_completed(futures)]

    success_count = sum(1 for _, r in results if not isinstance(r, str))
    print(f"Successfully processed {success_count} out of {len(test_cases)} test cases.")

except Exception as e:
    print(f"Critical error: {str(e)}")
    traceback.print_exc()

finally:
    end_time = time.time()
    print(f"Execution completed in {end_time - start_time:.2f} seconds.")

## Results

In [15]:
import pandas as pd
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

# Step 1: Create DataFrame from prediction results
pred_df = pd.DataFrame([
    {"index": idx, "pred_score": result.test_results[0].metrics_data[0].score, "reason": result.test_results[0].metrics_data[0].reason}
    for idx, result in results
    if not isinstance(result, str)
])

# Step 2: Join with ground truth labels
merged_df = pd.merge(pred_df, hallucination_df[["index", "hallucinated", "actual_output"]], on="index")

# Step 3: Round scores to get predicted labels
merged_df["pred_label"] = merged_df["pred_score"].round().astype(int)
merged_df["true_label"] = merged_df["hallucinated"].astype(int)

# Step 4: Calculate metrics
accuracy = accuracy_score(merged_df["true_label"], merged_df["pred_label"])
precision = precision_score(merged_df["true_label"], merged_df["pred_label"], zero_division=0)
recall = recall_score(merged_df["true_label"], merged_df["pred_label"], zero_division=0)
f1 = f1_score(merged_df["true_label"], merged_df["pred_label"], zero_division=0)

# Output
print(f"Accuracy: {accuracy:.3f}")
print(f"Precision: {precision:.3f}")
print(f"Recall: {recall:.3f}")
print(f"F1 Score: {f1:.3f}")

Accuracy: 0.653
Precision: 0.615
Recall: 0.681
F1 Score: 0.646


## G-Eval

In [43]:
import concurrent.futures
import time
import traceback
import pandas as pd
from deepeval.test_case import LLMTestCase, LLMTestCaseParams
from deepeval.metrics import GEval
from deepeval.evaluate import evaluate

# === Define GEval Hallucination Metric ===
hallucination_metric = GEval(
    name="Hallucination",
    model=claude_instance,
    criteria="Determine whether the actual output contains hallucinated information not properly grounded in the provided context.",
    evaluation_steps=[
        "Check whether any parts of the 'actual output' introduce facts, claims, or information that is not supported by or contradicts the 'context'.",
        "Penalize fabricated facts, statistics, dates, names, or details that are not found in the context.",
        "Allow reasonable paraphrasing and logical inferences directly supported by the context, but not invention of new information.",
        "Any unsupported information is considered a hallucination."
    ],
    evaluation_params=[
        LLMTestCaseParams.INPUT,
        LLMTestCaseParams.ACTUAL_OUTPUT,
        LLMTestCaseParams.CONTEXT
    ]
)

# === Create test cases from DataFrame ===
test_cases = [
    (row["index"], LLMTestCase(
        input=row["input"],
        actual_output=row["actual_output"],
        context=[row["context"]]
    ))
    for _, row in df.iterrows()
]

# === Evaluation function ===
def process_test_case(index, test_case, hallucination_metric):
    try:
        result = evaluate(
            test_cases=[test_case],
            metrics=[hallucination_metric],
            write_cache=False,
            print_results=False,
            display=None,
            show_indicator=False
        )
        return (index, result)
    except Exception as e:
        return (index, f"Error in test case {index}: {str(e)}")

start_time = time.time()

try:
    with concurrent.futures.ThreadPoolExecutor(max_workers=min(5, len(test_cases))) as executor:
        futures = [
            executor.submit(process_test_case, idx, case, hallucination_metric)
            for idx, case in test_cases
        ]
        results = [f.result() for f in concurrent.futures.as_completed(futures)]

    success_count = sum(1 for _, r in results if not isinstance(r, str))
    print(f"Successfully processed {success_count} out of {len(test_cases)} test cases.")

except Exception as e:
    print(f"Critical error: {str(e)}")
    traceback.print_exc()

finally:
    end_time = time.time()
    print(f"Execution completed in {end_time - start_time:.2f} seconds.")

✓ Tests finished 🎉! Run 'deepeval login' to save and analyze evaluation results on Confident AI.
 
✨👀 Looking for a place for your LLM test data to live 🏡❤️ ? Use Confident AI to get & share testing reports, 
experiment with models/prompts, and catch regressions for your LLM system. Just run 'deepeval login' in the CLI.

✓ Tests finished 🎉! Run 'deepeval login' to save and analyze evaluation results on Confident AI.
 
✨👀 Looking for a place for your LLM test data to live 🏡❤️ ? Use Confident AI to get & share testing reports, 
experiment with models/prompts, and catch regressions for your LLM system. Just run 'deepeval login' in the CLI.

✓ Tests finished 🎉! Run 'deepeval login' to save and analyze evaluation results on Confident AI.
 
✨👀 Looking for a place for your LLM test data to live 🏡❤️ ? Use Confident AI to get & share testing reports, 
experiment with models/prompts, and catch regressions for your LLM system. Just run 'deepeval login' in the CLI.

✓ Tests finished 🎉! Run 'deepeval login' to save and analyze evaluation results on Confident AI.
 
✨👀 Looking for a place for your LLM test data to live 🏡❤️ ? Use Confident AI to get & share testing reports, 
experiment with models/prompts, and catch regressions for your LLM system. Just run 'deepeval login' in the CLI.

✓ Tests finished 🎉! Run 'deepeval login' to save and analyze evaluation results on Confident AI.
 
✨👀 Looking for a place for your LLM test data to live 🏡❤️ ? Use Confident AI to get & share testing reports, 
experiment with models/prompts, and catch regressions for your LLM system. Just run 'deepeval login' in the CLI.

...

✓ Tests finished 🎉! Run 'deepeval login' to save and analyze evaluation results on Confident AI.
 
✨👀 Looking for a place for your LLM test data to live 🏡❤️ ? Use Confident AI to get & share testing reports, 
experiment with models/prompts, and catch regressions for your LLM system. Just run 'deepeval login' in the CLI.

✓ Tests finished 🎉! Run 'deepeval login' to save and analyze evaluation results on Confident AI.
 
✨👀 Looking for a place for your LLM test data to live 🏡❤️ ? Use Confident AI to get & share testing reports, 
experiment with models/prompts, and catch regressions for your LLM system. Just run 'deepeval login' in the CLI.

✓ Tests finished 🎉! Run 'deepeval login' to save and analyze evaluation results on Confident AI.
 
✨👀 Looking for a place for your LLM test data to live 🏡❤️ ? Use Confident AI to get & share testing reports, 
experiment with models/prompts, and catch regressions for your LLM system. Just run 'deepeval login' in the CLI.

✓ Tests finished 🎉! Run 'deepeval login' to save and analyze evaluation results on Confident AI.
 
✨👀 Looking for a place for your LLM test data to live 🏡❤️ ? Use Confident AI to get & share testing reports, 
experiment with models/prompts, and catch regressions for your LLM system. Just run 'deepeval login' in the CLI.

Successfully processed 1000 out of 1000 test cases.
Execution completed in 676.72 seconds.


In [44]:
import pandas as pd
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

# Step 1: Create DataFrame from prediction results
pred_df = pd.DataFrame([
    {"index": idx, "pred_score": 1 - result.test_results[0].metrics_data[0].score, "reason": result.test_results[0].metrics_data[0].reason}
    for idx, result in results
    if not isinstance(result, str)
])

# Step 2: Join with ground truth labels
merged_df = pd.merge(pred_df, hallucination_df[["index", "hallucinated", "actual_output"]], on="index")

# Step 3: Round scores to get predicted labels
merged_df["pred_label"] = merged_df["pred_score"].round().astype(int)
merged_df["true_label"] = merged_df["hallucinated"].astype(int)

# Step 4: Calculate metrics
accuracy = accuracy_score(merged_df["true_label"], merged_df["pred_label"])
precision = precision_score(merged_df["true_label"], merged_df["pred_label"], zero_division=0)
recall = recall_score(merged_df["true_label"], merged_df["pred_label"], zero_division=0)
f1 = f1_score(merged_df["true_label"], merged_df["pred_label"], zero_division=0)

# Output
print(f"Accuracy: {accuracy:.3f}")
print(f"Precision: {precision:.3f}")
print(f"Recall: {recall:.3f}")
print(f"F1 Score: {f1:.3f}")

Accuracy: 0.700
Precision: 0.946
Recall: 0.378
F1 Score: 0.540
